In [1]:
import pandas as pd
import json

# --- CONFIGURACIÓN DE RUTAS ---
# Formato: 'NombreDeLaCarpeta/NombreDelArchivo.json'

# Ruta 1: Los datos del producto (El que dices que la carpeta se llama meta_Electronics.json)
# Le agregamos .json al final del archivo por si Windows te está ocultando la extensión.
ruta_meta = 'meta_Electronics.json/meta_Electronics.json' 

# Ruta 2: Las reseñas de los usuarios (La carpeta Electronics)
ruta_reviews = 'Electronics/Electronics.json' 


# --- ELIGE CUÁL ANALIZAR PRIMERO (Cambiando la variable aquí) ---
# Vamos a probar primero con la metadata:
archivo_a_leer = ruta_meta 

print(f"Intentando leer: {archivo_a_leer}")

# INTENTO 1: Formato JSON Lines (Típico del dataset de Amazon)
try:
    df_muestra = pd.read_json(archivo_a_leer, lines=True, nrows=100000)
    print("✅ ¡Éxito! El archivo tiene formato JSON Lines.")

except Exception as e:
    print(f"⚠️ Falló JSON Lines. Error: {e}")
    print("⏳ Intentando cargar como JSON clásico...")
    
    # INTENTO 2: JSON Clásico
    with open(archivo_a_leer, 'r', encoding='utf-8') as file:
        data = json.load(file)
        df_muestra = pd.DataFrame(data[:100000])

# --- MOSTRAR RESULTADOS ---
print("\n--- 1. INFORMACIÓN DE LAS COLUMNAS ---")
df_muestra.info()

print("\n--- 2. VISTAZO A LOS DATOS (PRIMERAS 5 FILAS) ---")
display(df_muestra.head())

Intentando leer: meta_Electronics.json/meta_Electronics.json
✅ ¡Éxito! El archivo tiene formato JSON Lines.

--- 1. INFORMACIÓN DE LAS COLUMNAS ---
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   category         100000 non-null  object
 1   tech1            100000 non-null  str   
 2   description      100000 non-null  object
 3   fit              100000 non-null  str   
 4   title            100000 non-null  str   
 5   also_buy         100000 non-null  object
 6   tech2            100000 non-null  str   
 7   brand            100000 non-null  str   
 8   feature          100000 non-null  object
 9   rank             100000 non-null  object
 10  also_view        100000 non-null  object
 11  main_cat         100000 non-null  str   
 12  similar_item     100000 non-null  str   
 13  date             100000 non-null  str   
 14  price       

,category,tech1,description,fit,title,also_buy,tech2,brand,feature,rank,also_view,main_cat,similar_item,date,price,asin,imageURL,imageURLHighRes,details
0,"[Electronics, Camera &amp; Photo, Video Survei...",,[The following camera brands and models have b...,,Genuine Geovision 1 Channel 3rd Party NVR IP S...,[],,GeoVision,"[Genuine Geovision 1 Channel NVR IP Software, ...","[>#3,092 in Tools &amp; Home Improvement &gt; ...",[],Camera &amp; Photo,,"January 28, 2014",$65.00,0011300000,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN
1,"[Electronics, Camera &amp; Photo]",,[This second edition of the Handbook of Astron...,,"Books ""Handbook of Astronomical Image Processi...",[0999470906],,33 Books Co.,[Detailed chapters cover these fundamental top...,"[>#55,933 in Camera &amp; Photo (See Top 100 i...","[0943396670, 1138055360, 0999470906]",Camera &amp; Photo,,"June 17, 2003",,0043396828,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN
2,"[Electronics, eBook Readers &amp; Accessories,...",,[A zesty tale. (Publishers Weekly)<br /><br />...,,One Hot Summer,"[0425167798, 039914157X]",,Visit Amazon's Carolina Garcia Aguilera Page,[],"3,105,177 in Books (",[],Books,,,$11.49,0060009810,[],[],NaN
3,"[Electronics, eBook Readers & Accessories, eBo...",,[],,Hurray for Hattie Rabbit: Story and pictures (...,"[0060219521, 0060219580, 0060219394]",,Visit Amazon's Dick Gackenbach Page,[],"2,024,298 in Books (","[0060219521, 0060219475, 0060219394]",Books,,,.a-section.a-spacing-mini{margin-bottom:6px!im...,0060219602,[],[],NaN
4,"[Electronics, eBook Readers & Accessories, eBo...",,[&#8220;sex.lies.murder.fame. is brillllli&#82...,,sex.lies.murder.fame.: A Novel,[],,Visit Amazon's Lolita Files Page,[],"3,778,828 in Books (",[],Books,,,$13.95,0060786817,[],[],NaN


In [3]:
import pandas as pd

# 1. CARGAMOS LA MUESTRA (Usemos la de Reviews primero, que es la más grande)
# Recuerda ajustar la ruta si es necesario
ruta_reviews = 'Electronics.json/Electronics.json' 
df = pd.read_json(ruta_reviews, lines=True, nrows=100000)

print(f"Filas originales: {len(df)}")

# --- EL ATAJO DEL SENIOR (Borrar columnas inútiles para volumetría) ---
# En las reseñas, no necesitamos fotos de los usuarios ni formatos extraños.
columnas_a_borrar = ['image', 'style', 'reviewerName'] 
# Solo borramos las que realmente existan en el dataframe
columnas_borrar_existentes = [col for col in columnas_a_borrar if col in df.columns]
df = df.drop(columns=columnas_borrar_existentes)

# --- 2. ELIMINAR DUPLICADOS ---
# A veces el scraper de Amazon descargó la misma reseña dos veces
df = df.drop_duplicates()
print(f"Filas después de borrar duplicados: {len(df)}")

# --- 3. MANEJO DE NULOS (El dolor de cabeza de SQL) ---
# Si una reseña no tiene ID de producto (asin) o ID de usuario (reviewerID), es basura.
df = df.dropna(subset=['asin', 'reviewerID'])

# Para el texto de la reseña, si está vacío, le ponemos "Sin comentario" en vez de que SQL tire error
if 'reviewText' in df.columns:
    df['reviewText'] = df['reviewText'].fillna("Sin comentario")
    
# Para el resumen, igual
if 'summary' in df.columns:
    df['summary'] = df['summary'].fillna("Sin resumen")

print(f"Filas finales listas y limpias: {len(df)}")

# Vistazo final
display(df.head())

Filas originales: 100000
Filas después de borrar duplicados: 99956
Filas finales listas y limpias: 99956


,overall,verified,reviewTime,reviewerID,asin,reviewText,summary,unixReviewTime,vote
0,5,True,"07 17, 2002",A1N070NS9CJQ2I,0060009810,This was the first time I read Garcia-Aguilera...,Hit The Spot!,1026864000,NaN
1,5,False,"07 6, 2002",A3P0KRKOBQK1KN,0060009810,"As with all of Ms. Garcia-Aguilera's books, I ...",one hot summer is HOT HOT HOT!,1025913600,NaN
2,5,False,"07 3, 2002",A192HO2ICJ75VU,0060009810,I've not read any of Ms Aguilera's works befor...,One Hot Summer,1025654400,2
3,4,False,"06 30, 2002",A2T278FKFL3BLT,0060009810,This romance novel is right up there with the ...,I love this book!,1025395200,3
4,5,False,"06 28, 2002",A2ZUXVTW8RXBXW,0060009810,Carolina Garcia Aguilera has done it again. S...,One Hot Book,1025222400,NaN


In [4]:
# --- 4. ARREGLAR LOS VOTOS (NaN a 0) ---
if 'vote' in df.columns:
    # Reemplazamos NaN por 0. 
    # Además, Amazon a veces pone "1,200" con coma, la quitamos y convertimos a número entero.
    df['vote'] = df['vote'].fillna(0).astype(str).str.replace(',', '')
    df['vote'] = pd.to_numeric(df['vote'], errors='coerce').fillna(0).astype(int)

# --- 5. ARREGLAR LAS FECHAS PARA SQL SERVER ---
# SQL Server prefiere el formato YYYY-MM-DD. 
# Usamos unixReviewTime para crear una fecha perfecta y limpia.
df['reviewDate'] = pd.to_datetime(df['unixReviewTime'], unit='s')

# Borramos las columnas de fecha viejas que ya no necesitamos
df = df.drop(columns=['reviewTime', 'unixReviewTime'])

print("\n--- ¡DATOS 100% LISTOS PARA LA BASE DE DATOS! ---")
display(df.head())


--- ¡DATOS 100% LISTOS PARA LA BASE DE DATOS! ---


,overall,verified,reviewerID,asin,reviewText,summary,vote,reviewDate
0,5,True,A1N070NS9CJQ2I,0060009810,This was the first time I read Garcia-Aguilera...,Hit The Spot!,0,2002-07-17
1,5,False,A3P0KRKOBQK1KN,0060009810,"As with all of Ms. Garcia-Aguilera's books, I ...",one hot summer is HOT HOT HOT!,0,2002-07-06
2,5,False,A192HO2ICJ75VU,0060009810,I've not read any of Ms Aguilera's works befor...,One Hot Summer,2,2002-07-03
3,4,False,A2T278FKFL3BLT,0060009810,This romance novel is right up there with the ...,I love this book!,3,2002-06-30
4,5,False,A2ZUXVTW8RXBXW,0060009810,Carolina Garcia Aguilera has done it again. S...,One Hot Book,0,2002-06-28
